# Anomaly Detection of Time Series Data
## A Comparative Analysis of KNN, LOF and EWMA for Anomaly Detection Using Sliding Window Approaches

**Author:** Patel Mohamed Suhail (2584465) | **Supervisor:** Montaz Ali | Honours Research

This notebook reproduces every experiment in the report: baseline evaluation, parameter sensitivity (k, lambda, window size) and window-overlap sensitivity, on four labelled Numenta Anomaly Benchmark (NAB) series.

**Repository layout expected**
```
anomaly_detection.ipynb
nab_data/combined_windows_full.json     # NAB ground-truth anomaly windows
nab_data/raw/*.csv                      # the four NAB series
results/                                # created automatically
figures/                                # created automatically
```
**Requirements:** `numpy pandas scipy scikit-learn matplotlib jupyter`

In [ ]:
import json, time, tracemalloc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, roc_curve)

warnings.filterwarnings("ignore")
np.random.seed(42)

BASE = Path(".")
RAW = BASE / "nab_data" / "raw"
RESULTS = BASE / "results"; RESULTS.mkdir(exist_ok=True)
FIGS = BASE / "figures"; FIGS.mkdir(exist_ok=True)

with open(BASE / "nab_data" / "combined_windows_full.json") as f:
    ALL_WINDOWS = json.load(f)

DATASETS = {
    "nyc_taxi": {"file": RAW / "nyc_taxi.csv", "gt_key": "realKnownCause/nyc_taxi.csv"},
    "speed_7578": {"file": RAW / "speed_7578.csv", "gt_key": "realTraffic/speed_7578.csv"},
    "ec2_cpu_utilization_24ae8d": {"file": RAW / "ec2_cpu_utilization_24ae8d.csv",
                                   "gt_key": "realAWSCloudwatch/ec2_cpu_utilization_24ae8d.csv"},
    "Twitter_volume_AMZN": {"file": RAW / "Twitter_volume_AMZN.csv", "gt_key": "realTweets/Twitter_volume_AMZN.csv"},
}

DEFAULT_K, DEFAULT_LAMBDA, DEFAULT_WINDOW, DEFAULT_OVERLAP = 5, 0.2, 100, 0.5

## 1. Data loading and labelling

In [ ]:
def load_series(name):
    cfg = DATASETS[name]
    df = pd.read_csv(cfg["file"])
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    gt = [(pd.Timestamp(s), pd.Timestamp(e)) for s, e in ALL_WINDOWS[cfg["gt_key"]]]
    return df, gt

def point_labels(df, gt_windows):
    """1 if the timestamp lies inside any ground-truth anomaly window."""
    labels = np.zeros(len(df), dtype=int)
    for s, e in gt_windows:
        labels[((df["timestamp"] >= s) & (df["timestamp"] <= e)).values] = 1
    return labels

rows = []
for name in DATASETS:
    df, gt = load_series(name)
    pl = point_labels(df, gt)
    rows.append(dict(dataset=name, points=len(df), anomalous_points=int(pl.sum()),
                     anomalous_pct=round(100 * pl.mean(), 1), anomaly_windows=len(gt)))
pd.DataFrame(rows)

## 2. Sliding windows and feature extraction
Each window is summarised by its mean, standard deviation, skewness and kurtosis. A window is labelled anomalous when at least 5% of its points are anomalous.

In [ ]:
def make_windows(n, window_size, overlap):
    step = max(1, int(round(window_size * (1 - overlap))))
    return list(range(0, n - window_size + 1, step)), window_size

def window_features(values, starts, window_size):
    feats = np.zeros((len(starts), 4))
    for i, s in enumerate(starts):
        w = values[s:s + window_size]
        feats[i, 0] = np.mean(w)
        feats[i, 1] = np.std(w)
        feats[i, 2] = skew(w) if np.std(w) > 1e-12 else 0.0
        feats[i, 3] = kurtosis(w) if np.std(w) > 1e-12 else 0.0
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

def window_labels(point_lbls, starts, window_size, min_frac=0.05):
    return np.array([1 if point_lbls[s:s + window_size].mean() >= min_frac else 0 for s in starts])

def train_test_split_index(n, frac=0.7):          # temporal 70/30 split, no leakage
    return int(n * frac)

## 3. The three detectors
**KNN** – mean distance to the k nearest training windows. **LOF** – Local Outlier Factor in novelty mode. **EWMA** – raw-value exponentially weighted moving average with a 3-sigma control limit and a two-in-a-row rule.

In [ ]:
def run_knn(train_feat, test_feat, k):
    tracemalloc.start(); t0 = time.perf_counter()
    mu, sd = train_feat.mean(0), train_feat.std(0) + 1e-9
    tr, te = (train_feat - mu) / sd, (test_feat - mu) / sd
    nn = NearestNeighbors(n_neighbors=min(k, len(tr))).fit(tr)
    d_tr, _ = nn.kneighbors(tr, n_neighbors=min(k + 1, len(tr)))
    train_score = d_tr[:, 1:].mean(axis=1) if d_tr.shape[1] > 1 else d_tr[:, 0]
    d_te, _ = nn.kneighbors(te)
    test_score = d_te.mean(axis=1)
    el = time.perf_counter() - t0; peak = tracemalloc.get_traced_memory()[1]; tracemalloc.stop()
    return train_score, test_score, el, peak / 2**20

def run_lof(train_feat, test_feat, k):
    tracemalloc.start(); t0 = time.perf_counter()
    mu, sd = train_feat.mean(0), train_feat.std(0) + 1e-9
    tr, te = (train_feat - mu) / sd, (test_feat - mu) / sd
    k_eff = min(k, len(tr) - 1) if len(tr) > 1 else 1
    lof = LocalOutlierFactor(n_neighbors=max(1, k_eff), novelty=True).fit(tr)
    train_score, test_score = -lof.negative_outlier_factor_, -lof.score_samples(te)
    el = time.perf_counter() - t0; peak = tracemalloc.get_traced_memory()[1]; tracemalloc.stop()
    return train_score, test_score, el, peak / 2**20

def run_ewma(train_vals, test_vals, lam, window_size, overlap, n_sigma=3.0):
    tracemalloc.start(); t0 = time.perf_counter()

    def residuals(vals, ewma0):
        out, prev = np.zeros(len(vals)), ewma0
        for i, x in enumerate(vals):
            prev = lam * x + (1 - lam) * prev
            out[i] = prev
        return vals - out

    train_res = residuals(train_vals, train_vals[0])
    sigma = train_res.std() + 1e-9
    ucl = n_sigma * sigma
    test_res = residuals(test_vals, train_vals[-1])

    def two_in_a_row(res, limit):
        ex = np.abs(res) > limit
        flag = np.zeros(len(res), dtype=int)
        for i in range(1, len(res)):
            if ex[i] and ex[i - 1]:
                flag[i] = flag[i - 1] = 1
        return flag

    test_flags = two_in_a_row(test_res, ucl)
    te_starts, ws = make_windows(len(test_vals), window_size, overlap)
    wflag = np.array([1 if test_flags[s:s + ws].sum() > 0 else 0 for s in te_starts])
    wscore = np.array([np.max(np.abs(test_res[s:s + ws])) / sigma for s in te_starts])
    el = time.perf_counter() - t0; peak = tracemalloc.get_traced_memory()[1]; tracemalloc.stop()
    return wflag, wscore, el, peak / 2**20, len(te_starts)

## 4. Evaluation helpers
The alarm threshold for KNN and LOF is the 95th percentile of the training scores.

In [ ]:
def threshold_from_train(train_score, pct=95):
    return np.percentile(train_score, pct)

def evaluate(y_true, y_pred, y_score=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    auc = np.nan
    if y_score is not None and len(np.unique(y_true)) > 1:
        auc = roc_auc_score(y_true, y_score)
    return dict(TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn),
                precision=precision_score(y_true, y_pred, zero_division=0),
                recall=recall_score(y_true, y_pred, zero_division=0),
                f1=f1_score(y_true, y_pred, zero_division=0),
                specificity=tn / (tn + fp) if (tn + fp) else np.nan, auc=auc)

def build_dataset_windows(name, window_size, overlap):
    df, gt = load_series(name)
    values = df["value"].astype(float).values
    plbl = point_labels(df, gt)
    split = train_test_split_index(len(values))
    tr_v, te_v = values[:split], values[split:]
    tr_s, ws = make_windows(len(tr_v), window_size, overlap)
    te_s, _ = make_windows(len(te_v), window_size, overlap)
    return dict(train_vals=tr_v, test_vals=te_v, tr_starts=tr_s, te_starts=te_s, ws=ws,
                train_wlbl=window_labels(plbl[:split], tr_s, ws),
                test_wlbl=window_labels(plbl[split:], te_s, ws))

def run_one(name, k=DEFAULT_K, lam=DEFAULT_LAMBDA, window_size=DEFAULT_WINDOW,
            overlap=DEFAULT_OVERLAP, return_scores=False):
    d = build_dataset_windows(name, window_size, overlap)
    tr_f = window_features(d["train_vals"], d["tr_starts"], d["ws"])
    te_f = window_features(d["test_vals"], d["te_starts"], d["ws"])
    n_te = len(d["te_starts"]); rows, scores = [], {}

    for method, fn in (("KNN", run_knn), ("LOF", run_lof)):
        tr_s, te_s, t, mem = fn(tr_f, te_f, k)
        pred = (te_s > threshold_from_train(tr_s)).astype(int)
        m = evaluate(d["test_wlbl"], pred, te_s)
        m.update(dataset=name, method=method, k=k, lam=None, window=window_size, overlap=overlap,
                 ms_per_window=1000 * t / max(1, n_te), peak_mem_mb=mem, n_windows_test=n_te)
        rows.append(m); scores[method] = te_s

    flag, score, t, mem, n = run_ewma(d["train_vals"], d["test_vals"], lam, window_size, overlap)
    m = evaluate(d["test_wlbl"], flag, score)
    m.update(dataset=name, method="EWMA", k=None, lam=lam, window=window_size, overlap=overlap,
             ms_per_window=1000 * t / max(1, n), peak_mem_mb=mem, n_windows_test=n)
    rows.append(m); scores["EWMA"] = score
    return (rows, d["test_wlbl"], scores) if return_scores else rows

## 5. Baseline evaluation (default parameters)

In [ ]:
baseline_rows = []
for name in DATASETS:
    baseline_rows.extend(run_one(name))
baseline = pd.DataFrame(baseline_rows)
baseline.to_csv(RESULTS / "baseline.csv", index=False)
cols = ["dataset", "method", "TP", "FP", "FN", "TN", "precision", "recall", "f1", "specificity", "auc",
        "ms_per_window", "peak_mem_mb"]
baseline[cols].round(3)

### 5.1 Mean performance across datasets

In [ ]:
baseline.groupby("method")[["precision", "recall", "f1", "specificity", "auc", "ms_per_window", "peak_mem_mb"]].mean().round(3)

### 5.2 Confusion matrices

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(10, 13))
for i, name in enumerate(DATASETS):
    for j, method in enumerate(["KNN", "LOF", "EWMA"]):
        r = baseline[(baseline.dataset == name) & (baseline.method == method)].iloc[0]
        cm = np.array([[r.TN, r.FP], [r.FN, r.TP]])
        ax = axes[i, j]; ax.imshow(cm, cmap="Blues")
        for (a, b), v in np.ndenumerate(cm):
            ax.text(b, a, int(v), ha="center", va="center", fontsize=13,
                    color="white" if v > cm.max() / 2 else "black")
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["Normal", "Anomaly"]); ax.set_yticklabels(["Normal", "Anomaly"])
        ax.set_title(f"{method} - {name}", fontsize=8)
        if i == 3: ax.set_xlabel("Predicted")
        if j == 0: ax.set_ylabel("Actual")
plt.tight_layout(); plt.savefig(FIGS / "confusion_matrices.png", dpi=150); plt.show()

### 5.3 Precision, recall, F1 and AUC by dataset and method

In [ ]:
metrics = ["precision", "recall", "f1", "auc"]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, m in zip(axes, metrics):
    baseline.pivot(index="dataset", columns="method", values=m)[["KNN", "LOF", "EWMA"]].plot.bar(ax=ax, rot=30, legend=False)
    ax.set_title(m.upper() if m == "auc" else m.capitalize()); ax.set_xlabel(""); ax.set_ylim(0, 1.05)
axes[0].legend(title="Method"); plt.tight_layout()
plt.savefig(FIGS / "baseline_metrics.png", dpi=150); plt.show()

### 5.4 Runtime and memory

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, m, t in zip(axes, ["ms_per_window", "peak_mem_mb"], ["Runtime per test window (ms)", "Peak memory (MB)"]):
    baseline.pivot(index="dataset", columns="method", values=m)[["KNN", "LOF", "EWMA"]].plot.bar(ax=ax, rot=30)
    ax.set_title(t); ax.set_xlabel("")
plt.tight_layout(); plt.savefig(FIGS / "runtime_memory.png", dpi=150); plt.show()

### 5.5 ROC curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, name in zip(axes, DATASETS):
    _, y, sc = run_one(name, return_scores=True)
    for method, s in sc.items():
        if len(np.unique(y)) > 1:
            fpr, tpr, _ = roc_curve(y, s)
            ax.plot(fpr, tpr, label=f"{method} (AUC={roc_auc_score(y, s):.2f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_title(name, fontsize=9)
    ax.set_xlabel("False positive rate"); ax.legend(fontsize=7)
axes[0].set_ylabel("True positive rate"); plt.tight_layout()
plt.savefig(FIGS / "roc_curves.png", dpi=150); plt.show()

## 6. Parameter sensitivity
One parameter is varied at a time; all others stay at their defaults. k applies to KNN/LOF, lambda to EWMA, and window size to all three methods.

In [ ]:
K_VALUES, LAMBDA_VALUES, WINDOW_VALUES = [3, 5, 10, 20], [0.05, 0.1, 0.2, 0.4], [50, 100, 200]
sens_rows = []
for name in DATASETS:
    for k in K_VALUES:
        for r in run_one(name, k=k):      r.update(sweep="k", sweep_value=k);       sens_rows.append(r)
    for lam in LAMBDA_VALUES:
        for r in run_one(name, lam=lam):  r.update(sweep="lambda", sweep_value=lam); sens_rows.append(r)
    for w in WINDOW_VALUES:
        for r in run_one(name, window_size=w): r.update(sweep="window", sweep_value=w); sens_rows.append(r)
sens = pd.DataFrame(sens_rows)
sens.to_csv(RESULTS / "sensitivity.csv", index=False)

def sweep_table(df, sweep, methods):
    d = df[(df.sweep == sweep) & (df.method.isin(methods))]
    return d.pivot_table(index="sweep_value", columns="method", values="f1", aggfunc="mean").round(3)

print("Mean F1 vs k");      display(sweep_table(sens, "k", ["KNN", "LOF"]))
print("Mean F1 vs lambda"); display(sweep_table(sens, "lambda", ["EWMA"]))
print("Mean F1 vs window"); display(sweep_table(sens, "window", ["KNN", "LOF", "EWMA"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, (sw, ms, xl) in zip(axes, [("k", ["KNN", "LOF"], "k"), ("lambda", ["EWMA"], "lambda"),
                                    ("window", ["KNN", "LOF", "EWMA"], "window size")]):
    sweep_table(sens, sw, ms).plot(ax=ax, marker="o"); ax.set_xlabel(xl); ax.set_ylabel("Mean F1")
    ax.set_title(f"Mean F1 vs {xl}"); ax.set_ylim(0, 1)
plt.tight_layout(); plt.savefig(FIGS / "parameter_sensitivity.png", dpi=150); plt.show()

## 7. Window-overlap sensitivity

In [ ]:
OVERLAP_VALUES = [0.0, 0.25, 0.5, 0.75]
ov_rows = []
for name in DATASETS:
    for ov in OVERLAP_VALUES:
        for r in run_one(name, overlap=ov):
            r.update(sweep="overlap", sweep_value=ov); ov_rows.append(r)
overlap = pd.DataFrame(ov_rows)
overlap.to_csv(RESULTS / "overlap.csv", index=False)

tbl = overlap.pivot_table(index="sweep_value", columns="method", values="f1", aggfunc="mean").round(3)[["KNN", "LOF", "EWMA"]]
display(tbl)
ax = tbl.plot(marker="o", figsize=(6, 3.8)); ax.set_xlabel("Window overlap"); ax.set_ylabel("Mean F1"); ax.set_ylim(0, 1)
plt.tight_layout(); plt.savefig(FIGS / "overlap_sensitivity.png", dpi=150); plt.show()

## 8. Summary of findings
* **KNN** gave the most consistent overall performance and the highest mean F1 and AUC.
* **LOF** was strongest on the noisy, multi-density cloud series but weakest on series with few anomalies.
* **EWMA** was perfect on the series with an abrupt level shift but detected nothing on the strongly periodic taxi series.
* All methods were cheap (well under 1 ms per window, under 1 MB); longer windows and more overlap generally helped the window-based methods.

**Reference papers (from the research proposal):** Cover & Hart (1967); Keogh, Lin & Fu (2005); Li et al. (2021); Lucas & Saccucci (1990); Pokrajac, Lazarevic & Latecki (2007); Salehi et al. (2016); Teng (2010); Tian et al. (2014).